# Diário da Tarefa 5 — Árvore final e teste único

**Período:** 12/10/2026 a 25/10/2026  
**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)

**Equipe:** Breno Igor Ribeiro Da Silva; Isaque Ferreira de Jesus; Lukas Paulo Alves de Carvalho; Renata Mayumi Iuvata; Ruan Abner Rodrigues Morales  
**Scrum Master da tarefa:** Breno Ribeiro
**Repositório GitHub:** https://github.com/RedSwann/sistema-preditivo-de-falhas-em-redes

> A árvore que entra aqui é a da Tarefa 4. O baseline e o rótulo são os da Tarefa 2. O teste, guardado desde a Tarefa 2, é medido uma vez.
>
> Continua sendo uma árvore de decisão. O grupo entrega as regras, a matriz e o que a árvore faz num fluxo que ela não viu — desde que esse fluxo tenha a própria ficha de baseline, calculada só no período inicial dele.

## Contrato desta tarefa

| | Artefato | Origem / destino |
|---|---|---|
| **Entra** | Corte da Tarefa 2 e ficha de baseline | Tarefa 2 |
| **Entra** | Árvore ajustada e regras | Tarefa 4 |
| **Sai** | Teste único: matriz 3×3, F1 por classe, F1 macro | Entrega |
| **Sai** | Checagem de fluxo não visto, com baseline próprio | Entrega |
| **Sai** | Ficha da árvore e dicionário v0.4 igual às colunas usadas | Entrega |

Não há tarefa seguinte.

- [x] A profundidade não foi escolhida de novo olhando o teste
- [x] O baseline do teste não usou medição do próprio teste

In [9]:
%pip install scikit-learn==1.6.1

In [10]:

from pathlib import Path
import json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix, classification_report, f1_score,
                             precision_recall_fscore_support, accuracy_score)
from sklearn.model_selection import ParameterGrid

CLASSES = ["OK", "RISCO", "FALHA"]
FEATURES = ["z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
            "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"]
RANDOM_STATE = 42

# Google Colab: utiliza a mesma pasta do Google Drive definida na Tarefa 2.
# Não copia nem altera os CSVs; apenas lê os arquivos já gerados.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
except ImportError:
    # Alternativa para execução local, mantendo a mesma estrutura de pastas.
    PASTA_PROJETO = Path.cwd() / "ripe_atlas"

PROCESSED = PASTA_PROJETO / "processed"
DATASET_PATH = PROCESSED / "dataset_B_com_recorte_temporal.csv"
BASELINE_PATH = PROCESSED / "baseline_por_fluxo.csv"
ARTEFATOS = PROCESSED / "artefatos_notebooks"
ARTEFATOS.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Não encontrei {DATASET_PATH}. Confira se a pasta ripe_atlas está no Meu Drive "
        "e se a Tarefa 2 já gerou dataset_B_com_recorte_temporal.csv."
    )

df = pd.read_csv(DATASET_PATH)
obrigatorias = FEATURES + ["classe", "bloco_temporal", "fluxo_id", "timestamp_utc"]
faltantes = [c for c in obrigatorias if c not in df.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes no dataset: {faltantes}")
df["classe"] = df["classe"].astype(str).str.upper()
df = df[df["classe"].isin(CLASSES)].copy()

treino = df[df["bloco_temporal"].eq("treino")].copy()
validacao = df[df["bloco_temporal"].eq("validacao")].copy()
teste = df[df["bloco_temporal"].eq("teste")].copy()

print("Google Drive:", PASTA_PROJETO)
print("Dataset carregado:", DATASET_PATH)
print("Registros:", len(df))
print("Treino/validação/teste:", len(treino), len(validacao), len(teste))
print("Classes:", df["classe"].value_counts().reindex(CLASSES, fill_value=0).to_dict())

def criar_pipeline(criterion="gini", max_depth=5, min_samples_leaf=25):
    # A imputação é ajustada somente no conjunto usado em fit.
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("arvore", DecisionTreeClassifier(
            criterion=criterion, max_depth=max_depth,
            min_samples_leaf=min_samples_leaf, random_state=RANDOM_STATE
        ))
    ])

def mostrar_metricas(y_true, y_pred, titulo):
    print(f"\n{titulo}")
    print("Matriz de confusão (linhas=real; colunas=previsto):")
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    display(pd.DataFrame(cm, index=[f"Real {c}" for c in CLASSES],
                         columns=[f"Previsto {c}" for c in CLASSES]))
    print(classification_report(y_true, y_pred, labels=CLASSES,
                                target_names=CLASSES, zero_division=0))
    print("F1 macro:", round(f1_score(y_true, y_pred, labels=CLASSES,
                                      average="macro", zero_division=0), 4))
    return cm

def metricas_resumo(y, pred):
    p, r, f, _ = precision_recall_fscore_support(
        y, pred, labels=CLASSES, zero_division=0)
    return {"Acurácia": accuracy_score(y, pred),
            "F1 macro": f1_score(y, pred, labels=CLASSES,
                                  average="macro", zero_division=0),
            "Recall FALHA": r[CLASSES.index("FALHA")],
            "Recall RISCO": r[CLASSES.index("RISCO")]}

def regras_em_portugues(modelo, nomes_features=FEATURES, max_regras=12):
    arv = modelo.named_steps["arvore"]
    nomes = list(nomes_features)
    texto = export_text(arv, feature_names=nomes, decimals=3, show_weights=True)
    print(texto)
    tree = arv.tree_
    regras = []
    def percorrer(no, condicoes):
        feature = tree.feature[no]
        if feature != -2:
            nome = nomes[feature]
            limiar = tree.threshold[no]
            percorrer(tree.children_left[no], condicoes + [f"{nome} ≤ {limiar:.3f}"])
            percorrer(tree.children_right[no], condicoes + [f"{nome} > {limiar:.3f}"])
        else:
            classe = str(arv.classes_[int(np.argmax(tree.value[no][0]))])
            regras.append("SE " + " E ".join(condicoes) + f" ENTÃO {classe}")
    percorrer(0, [])
    for i, regra in enumerate(regras[:max_regras], 1):
        print(f"{i}. {regra}")
    return regras

def resumo_arvore(modelo):
    arv = modelo.named_steps["arvore"]
    return {"criterion": arv.criterion, "max_depth_configurado": arv.max_depth,
            "profundidade_real": arv.get_depth(), "folhas": arv.get_n_leaves(),
            "min_samples_leaf": arv.min_samples_leaf, "random_state": RANDOM_STATE}

def converter_json(obj):
    """Converte tipos NumPy/Pandas para tipos nativos aceitos pelo JSON."""
    if isinstance(obj, dict):
        return {str(k): converter_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [converter_json(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if pd.isna(obj) if not isinstance(obj, (dict, list, tuple, np.ndarray)) else False:
        return None
    return obj

# Colunas de apoio para auditar distância do caminho sem fornecê-las à árvore.
if "mediana" not in df.columns:
    raise ValueError("A coluna mediana é necessária para identificar caminhos curtos/longos.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive: /content/drive/MyDrive/ripe_atlas
Dataset carregado: /content/drive/MyDrive/ripe_atlas/processed/dataset_B_com_recorte_temporal.csv
Registros: 34064
Treino/validação/teste: 17028 6806 10230
Classes: {'OK': 26274, 'RISCO': 3848, 'FALHA': 3942}


## 1. Árvore que será medida

- [x] Critério, profundidade, mínimo de amostras na folha, colunas
- [x] Três regras em português, as mesmas da Tarefa 4 ou as revistas antes de abrir o teste
- [x] Confirmação de que país, IP, rota e RTT absoluto não estão na árvore

**Parâmetros:**

criterion = gini
max_depth = 4
min_samples_leaf = 10
random_state = 42

Profundidade real: 4
Quantidade de folhas: 11

**Colunas:**
z_robusto, aumento_pct, jitter_relativo, perda_pct, timeout_atual, n5_timeout, n5_aumento80 e n5_risco.  

**Regras:**
1. SE z_robusto ≤ 1.996 E jitter_relativo ≤ 3.003 E perda_pct ≤ 16.667 ENTÃO OK.

2. SE z_robusto ≤ 1.996 E jitter_relativo ≤ 3.003 E perda_pct > 16.667 ENTÃO FALHA.

3. SE z_robusto ≤ 1.996 E jitter_relativo > 3.003 E n5_risco ≤ 1.500 E z_robusto ≤ -8.620 ENTÃO OK.

In [11]:
ARQ_T4 = ARTEFATOS / "arvore_tarefa4.joblib"
ARQ_PARAM_T4 = ARTEFATOS / "parametros_tarefa4.json"
ARQ_REGRAS_T4 = ARTEFATOS / "regras_tarefa4.csv"
if not ARQ_T4.exists() or not ARQ_PARAM_T4.exists():
    raise FileNotFoundError("Execute primeiro o notebook Tarefa4_Ajuste_da_Arvore.ipynb.")

# Congela exatamente a árvore produzida na Tarefa 4; não há nova busca de parâmetros.
modelo_final = joblib.load(ARQ_T4)
with open(ARQ_PARAM_T4, encoding="utf-8") as f:
    config_t4 = json.load(f)
print("Parâmetros congelados:", config_t4["parametros"])
print("Resumo:", resumo_arvore(modelo_final))
print("Features:", FEATURES)
proibidas = {"fluxo_id", "rtt_ms", "dst_addr", "src_addr", "dst_name",
             "destino_pais_informado", "pais_origem_esperado", "pais", "IP", "rota_id"}
print("Features proibidas presentes:", sorted(set(FEATURES) & proibidas))
if set(FEATURES) & proibidas:
    raise AssertionError("Há coluna proibida entre as features da árvore.")
print("Confirmação: país, IP, rota, fluxo_id e RTT absoluto não são preditores.")
regras_final = pd.read_csv(ARQ_REGRAS_T4)["regra"].dropna().astype(str).tolist()
print("\nTrês primeiras regras da árvore final (mantidas da Tarefa 4):")
for i, regra in enumerate(regras_final[:3], 1):
    print(f"{i}. {regra}")
if len(regras_final) < 3:
    print(f"A árvore possui somente {len(regras_final)} regra(s) terminal(is).")


Parâmetros congelados: {'criterion': 'gini', 'max_depth': 4, 'min_samples_leaf': 10}
Resumo: {'criterion': 'gini', 'max_depth_configurado': 4, 'profundidade_real': 4, 'folhas': np.int64(11), 'min_samples_leaf': 10, 'random_state': 42}
Features: ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']
Features proibidas presentes: []
Confirmação: país, IP, rota, fluxo_id e RTT absoluto não são preditores.

Três primeiras regras da árvore final (mantidas da Tarefa 4):
1. SE z_robusto ≤ 1.996 E jitter_relativo ≤ 3.003 E perda_pct ≤ 16.667 ENTÃO OK
2. SE z_robusto ≤ 1.996 E jitter_relativo ≤ 3.003 E perda_pct > 16.667 ENTÃO FALHA
3. SE z_robusto ≤ 1.996 E jitter_relativo > 3.003 E n5_risco ≤ 1.500 E z_robusto ≤ -8.620 ENTÃO OK


## 2. Teste temporal (uma vez)

Fluxos conhecidos, período mais recente do Período B.

- [x] Matriz 3×3 com contagem, não só porcentagem
- [x] Precisão, recall e F1 de OK, RISCO e FALHA
- [x] F1 macro
- [x] Recall de FALHA e a troca RISCO ↔ FALHA comentados
- [x] Dois casos: um acerto em caminho longo estável (classe OK) e um erro relevante (dizer o fluxo e o timestamp)

**Matriz de confusão (linhas = classe real; colunas = prevista):**

| Classe real | Previsto OK | Previsto RISCO | Previsto FALHA |
|---|---:|---:|---:|
| OK | 8.206 | 0 | 0 |
| RISCO | 0 | 968 | 0 |
| FALHA | 3 | 1 | 1.052 |

**F1 macro no teste:** 0,9991. **Acurácia:** 99,9609%. Foram 4 erros em 10.230 registros.

**Recall de FALHA:** 99,62% (1.052 de 1.056). Houve 3 casos FALHA → OK e 1 caso FALHA → RISCO; não houve confusão RISCO → FALHA. A matriz e o relatório por classe também são exibidos na célula de avaliação.

**Casos auditados:**
- Acerto em caminho longo estável: fluxo `6811|54.39.185.31`, em `2026-09-28 04:47:19 UTC`; classe real e prevista **OK**.
- Erro relevante: fluxo `7423|103.52.117.1`, em `2026-09-28 05:16:32 UTC`; classe real **FALHA**, prevista **OK**.

In [12]:
# ÚNICA avaliação do teste temporal. Não usar os resultados para alterar parâmetros.
X_teste, y_teste = teste[FEATURES], teste["classe"]
pred_teste = modelo_final.predict(X_teste)
cm_teste = mostrar_metricas(y_teste, pred_teste, "Tarefa 5 — TESTE TEMPORAL (avaliação única)")

resultado_teste = teste.copy()
resultado_teste["classe_prevista"] = pred_teste
resultado_teste["acertou"] = resultado_teste["classe"].eq(
    resultado_teste["classe_prevista"])
metricas_teste = metricas_resumo(y_teste, pred_teste)
print(f"Acurácia: {metricas_teste['Acurácia']:.4%}")
print("Métricas principais:", {k: round(float(v), 4) for k,v in metricas_teste.items()})
print("Trocas RISCO ↔ FALHA (linhas=real; colunas=previsto):")
trocas_teste = pd.crosstab(resultado_teste["classe"], resultado_teste["classe_prevista"])
trocas_teste = trocas_teste.reindex(index=CLASSES, columns=CLASSES, fill_value=0)
display(trocas_teste)
recall_falha = metricas_teste["Recall FALHA"]
print(f"Recall da classe FALHA: {recall_falha:.4f}")
print("Interpretação automática:")
print(f"- De todas as FALHA reais, a árvore identificou {recall_falha:.1%}.")
print(f"- FALHA prevista como RISCO: {int(trocas_teste.loc['FALHA', 'RISCO'])} caso(s).")
print(f"- RISCO previsto como FALHA: {int(trocas_teste.loc['RISCO', 'FALHA'])} caso(s).")
print("- A diagonal representa acertos; as células fora da diagonal mostram confusões entre classes.")


Tarefa 5 — TESTE TEMPORAL (avaliação única)
Matriz de confusão (linhas=real; colunas=previsto):


,Previsto OK,Previsto RISCO,Previsto FALHA
Real OK,8206,0,0
Real RISCO,0,968,0
Real FALHA,3,1,1052


              precision    recall  f1-score   support

          OK       1.00      1.00      1.00      8206
       RISCO       1.00      1.00      1.00       968
       FALHA       1.00      1.00      1.00      1056

    accuracy                           1.00     10230
   macro avg       1.00      1.00      1.00     10230
weighted avg       1.00      1.00      1.00     10230

F1 macro: 0.9991
Acurácia: 99.9609%
Métricas principais: {'Acurácia': 0.9996, 'F1 macro': 0.9991, 'Recall FALHA': 0.9962, 'Recall RISCO': 1.0}
Trocas RISCO ↔ FALHA (linhas=real; colunas=previsto):


classe_prevista,OK,RISCO,FALHA
classe,,,
OK,8206,0,0
RISCO,0,968,0
FALHA,3,1,1052


Recall da classe FALHA: 0.9962
Interpretação automática:
- De todas as FALHA reais, a árvore identificou 99.6%.
- FALHA prevista como RISCO: 1 caso(s).
- RISCO previsto como FALHA: 0 caso(s).
- A diagonal representa acertos; as células fora da diagonal mostram confusões entre classes.


In [13]:
# Auditoria de exemplos do teste: caminho longo estável, caminho curto e erro relevante.
# A mediana é usada APENAS para descrever o caminho; não é feature do modelo.
medianas_teste = resultado_teste.groupby("fluxo_id")["mediana"].first().dropna()
if medianas_teste.empty:
    raise ValueError("Não há medianas de baseline disponíveis para caracterizar os caminhos.")
q25, q75 = medianas_teste.quantile([0.25, 0.75])
resultado_teste["mediana_caminho"] = resultado_teste["fluxo_id"].map(medianas_teste)
resultado_teste["tipo_caminho"] = np.select(
    [resultado_teste["mediana_caminho"] <= q25,
     resultado_teste["mediana_caminho"] >= q75],
    ["curto", "longo"], default="intermediario")

acerto_longo_ok = resultado_teste[
    (resultado_teste["classe"] == "OK") &
    (resultado_teste["classe_prevista"] == "OK") &
    (resultado_teste["tipo_caminho"] == "longo")]
erro_relevante = resultado_teste[
    (~resultado_teste["acertou"]) &
    (resultado_teste["classe"].isin(["RISCO", "FALHA"]) |
     resultado_teste["classe_prevista"].isin(["RISCO", "FALHA"]))]
exemplo_curto = resultado_teste[
    (resultado_teste["tipo_caminho"] == "curto")]

# Escolha determinística: primeiro registro cronológico de cada conjunto.
cols_auditoria = ["fluxo_id", "timestamp_utc", "mediana_caminho", "tipo_caminho",
                  "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
                  "classe", "classe_prevista", "acertou"]

def mostrar_exemplo(titulo, subconjunto):
    print("\\n" + titulo)
    if subconjunto.empty:
        print("Não foi encontrado exemplo que satisfaça este critério no teste.")
        return pd.DataFrame(columns=cols_auditoria)
    # Cria a coluna temporária no próprio subconjunto antes de ordenar.
    escolhido = subconjunto.assign(
        _timestamp_ord=pd.to_datetime(subconjunto["timestamp_utc"],
                                      errors="coerce", utc=True)
    ).sort_values("_timestamp_ord", na_position="last").head(1)
    escolhido = escolhido.drop(columns="_timestamp_ord")
    cols_existentes = [col for col in cols_auditoria if col in escolhido.columns]
    display(escolhido[cols_existentes])
    return escolhido[cols_existentes]

exemplo_longo_df = mostrar_exemplo("Acerto OK em caminho longo:", acerto_longo_ok)
exemplo_erro_df = mostrar_exemplo("Erro relevante:", erro_relevante)
exemplo_curto_df = mostrar_exemplo("Exemplo de caminho curto (acerto ou erro):", exemplo_curto)
exemplos_auditoria = pd.concat([
    exemplo_longo_df.assign(tipo_exemplo="acerto_OK_caminho_longo"),
    exemplo_erro_df.assign(tipo_exemplo="erro_relevante"),
    exemplo_curto_df.assign(tipo_exemplo="caminho_curto")
], ignore_index=True)
exemplos_auditoria.to_csv(ARTEFATOS / "casos_auditoria_teste.csv", index=False)
print("\nCritério de caminho: quartil inferior da mediana de baseline = curto; quartil superior = longo.")


\nAcerto OK em caminho longo:


,fluxo_id,timestamp_utc,mediana_caminho,tipo_caminho,z_robusto,aumento_pct,jitter_relativo,perda_pct,classe,classe_prevista,acertou
7934,6811|54.39.185.31,2026-09-28 04:47:19+00:00,294.912052,longo,-0.395309,-0.014381,0.160461,0.0,OK,OK,True


\nErro relevante:


,fluxo_id,timestamp_utc,mediana_caminho,tipo_caminho,z_robusto,aumento_pct,jitter_relativo,perda_pct,classe,classe_prevista,acertou
27260,7423|103.52.117.1,2026-09-28 05:16:32+00:00,405.960532,longo,2.637641,16.549827,0.001218,33.333333,FALHA,OK,False


\nExemplo de caminho curto (acerto ou erro):


,fluxo_id,timestamp_utc,mediana_caminho,tipo_caminho,z_robusto,aumento_pct,jitter_relativo,perda_pct,classe,classe_prevista,acertou
1496,6410|45.11.105.136,2026-09-28 04:36:56+00:00,1.342342,curto,1.247651,9.991641,2.822105,0.0,OK,OK,True



Critério de caminho: quartil inferior da mediana de baseline = curto; quartil superior = longo.


## 3. Avaliação adicional em fluxo excluído do treinamento

A divisão temporal oficial (treino/validação/teste) é mantida sem alterações. Como os mesmos fluxos aparecem nos três blocos, esta seção cria **uma avaliação auxiliar**: seleciona deterministicamente um fluxo com baseline A válido (mínimo de 1.500 RTTs) e registros no teste, retira todas as linhas desse fluxo do treino auxiliar e avalia uma árvore auxiliar somente nos registros de teste desse fluxo.

- [x] O fluxo selecionado não participa do ajuste da árvore auxiliar.
- [x] Mediana e MAD vêm do baseline do Período A; não são recalculadas com o trecho avaliado.
- [x] Os hiperparâmetros ficam congelados conforme a Tarefa 4.
- [x] A validação original da Tarefa 4 pode conter o fluxo selecionado; portanto, a escolha dos hiperparâmetros pode ter sido influenciada por ele. Esta é uma checagem fora do treinamento, não uma avaliação totalmente independente de toda a seleção.

**Importante:** esta análise é complementar. Ela não substitui nem altera a avaliação temporal oficial da seção 2, não modifica sua acurácia e não usa o resultado do teste para reajustar a árvore oficial.

**Fluxo reservado e resultados:** preenchidos automaticamente na célula seguinte e no arquivo `relatorio_fluxo_nao_visto.json`.


In [14]:
# Experimento adicional: fluxo inteiro retirado do treinamento auxiliar.
# A avaliação temporal oficial acima permanece intacta.
# O baseline é lido exclusivamente do Período A; não recalculamos mediana/MAD
# usando medições do fluxo avaliado.

if not BASELINE_PATH.exists():
    raise FileNotFoundError("baseline_por_fluxo.csv não foi encontrado.")

baselines = pd.read_csv(BASELINE_PATH)
colunas_base = {"fluxo_id", "status_baseline", "n_rtt_validos_A", "mediana", "MAD"}
if not colunas_base.issubset(baselines.columns):
    raise ValueError(
        f"Colunas ausentes em baseline_por_fluxo.csv: "
        f"{sorted(colunas_base - set(baselines.columns))}"
    )

# Mantém apenas fluxos com baseline válido e pelo menos 1.500 RTTs válidos no A.
elegiveis = baselines[
    baselines["status_baseline"].eq("baseline_ok")
    & (pd.to_numeric(baselines["n_rtt_validos_A"], errors="coerce") >= 1500)
].copy()
elegiveis["fluxo_id_str"] = elegiveis["fluxo_id"].astype(str)

# O fluxo precisa ter registros no teste para que a avaliação seja possível.
fluxos_com_teste = set(teste["fluxo_id"].astype(str))
candidatos = elegiveis[
    elegiveis["fluxo_id_str"].isin(fluxos_com_teste)
].copy()

relatorio_fluxo = {
    "status": "nao_avaliado",
    "fluxo_id": None,
    "n_rtt_validos_A": None,
    "mediana_A": None,
    "MAD_A": None,
    "n_treino_original": None,
    "n_treino_auxiliar": None,
    "n_validacao": None,
    "n_teste": 0,
    "acuracia": None,
    "f1_macro": None,
    "recall_falha": None,
    "matriz_confusao": None,
    "observacao_metodologica": ""
}

if candidatos.empty:
    print(
        "Não há fluxo com baseline A válido (>=1.500 RTTs) e registros no teste. "
        "A avaliação adicional não pode ser realizada."
    )
    relatorio_fluxo["observacao_metodologica"] = (
        "Não foi encontrado fluxo elegível com registros no teste."
    )
else:
    # Seleção determinística pelo identificador, sem consultar classe ou desempenho
    # dos registros de teste. No conjunto atual, a seleção esperada é 6410|45.11.105.136.
    base_fluxo = candidatos.sort_values("fluxo_id_str").iloc[0]
    fluxo_novo = str(base_fluxo["fluxo_id_str"])

    # Exclui TODAS as linhas desse fluxo do treino usado no ajuste auxiliar.
    treino_auxiliar = treino[
        treino["fluxo_id"].astype(str) != fluxo_novo
    ].copy()
    teste_fluxo = teste[
        teste["fluxo_id"].astype(str) == fluxo_novo
    ].copy()

    n_treino_original = int(
        (treino["fluxo_id"].astype(str) == fluxo_novo).sum()
    )
    n_validacao_fluxo = int(
        (validacao["fluxo_id"].astype(str) == fluxo_novo).sum()
    )

    print("Fluxo reservado para a avaliação adicional:", fluxo_novo)
    print("RTTs válidos no baseline A:", int(base_fluxo["n_rtt_validos_A"]))
    print("Mediana do baseline A:", float(base_fluxo["mediana"]))
    print("MAD do baseline A:", float(base_fluxo["MAD"]))
    print("Registros desse fluxo no treino original:", n_treino_original)
    print("Registros desse fluxo na validação original:", n_validacao_fluxo)
    print("Registros desse fluxo no teste:", len(teste_fluxo))
    print("Registros restantes no treino auxiliar:", len(treino_auxiliar))

    if treino_auxiliar.empty or teste_fluxo.empty:
        print("Não há dados suficientes para ajustar/avaliar a árvore auxiliar.")
        relatorio_fluxo.update({
            "status": "sem_dados_suficientes",
            "fluxo_id": fluxo_novo,
            "n_rtt_validos_A": int(base_fluxo["n_rtt_validos_A"]),
            "mediana_A": float(base_fluxo["mediana"]),
            "MAD_A": float(base_fluxo["MAD"]),
            "n_treino_original": n_treino_original,
            "n_treino_auxiliar": int(len(treino_auxiliar)),
            "n_validacao": n_validacao_fluxo,
            "n_teste": int(len(teste_fluxo)),
            "observacao_metodologica": (
                "Fluxo selecionado, mas sem dados suficientes para ajuste ou avaliação."
            )
        })
    else:
        # Hiperparâmetros congelados da Tarefa 4; nenhum ajuste é feito com o teste.
        modelo_fluxo_novo = criar_pipeline(**config_t4["parametros"])
        modelo_fluxo_novo.fit(
            treino_auxiliar[FEATURES],
            treino_auxiliar["classe"]
        )

        pred_fluxo = modelo_fluxo_novo.predict(teste_fluxo[FEATURES])
        metricas_fluxo = metricas_resumo(teste_fluxo["classe"], pred_fluxo)
        cm_fluxo = mostrar_metricas(
            teste_fluxo["classe"],
            pred_fluxo,
            "Avaliação adicional — fluxo excluído do treinamento"
        )

        fluxo_resultado = teste_fluxo.copy()
        fluxo_resultado["classe_prevista"] = pred_fluxo
        fluxo_resultado["acertou"] = (
            fluxo_resultado["classe"].eq(fluxo_resultado["classe_prevista"])
        )

        print("\nDistribuição real das classes no fluxo reservado:")
        display(fluxo_resultado["classe"].value_counts().reindex(CLASSES, fill_value=0))
        print("Matriz de confusão do fluxo reservado:")
        display(pd.DataFrame(
            cm_fluxo,
            index=[f"Real {c}" for c in CLASSES],
            columns=[f"Previsto {c}" for c in CLASSES]
        ))

        colunas_exibicao = [
            c for c in [
                "fluxo_id", "timestamp_utc", "classe",
                "classe_prevista", "acertou"
            ] if c in fluxo_resultado.columns
        ]
        display(fluxo_resultado[colunas_exibicao].head(20))

        relatorio_fluxo.update({
            "status": "avaliado",
            "fluxo_id": fluxo_novo,
            "n_rtt_validos_A": int(base_fluxo["n_rtt_validos_A"]),
            "mediana_A": float(base_fluxo["mediana"]),
            "MAD_A": float(base_fluxo["MAD"]),
            "n_treino_original": n_treino_original,
            "n_treino_auxiliar": int(len(treino_auxiliar)),
            "n_validacao": n_validacao_fluxo,
            "n_teste": int(len(teste_fluxo)),
            "acuracia": float(metricas_fluxo["Acurácia"]),
            "f1_macro": float(metricas_fluxo["F1 macro"]),
            "recall_falha": float(metricas_fluxo["Recall FALHA"]),
            "matriz_confusao": cm_fluxo.tolist(),
            "observacao_metodologica": (
                "Todas as linhas do fluxo foram excluídas do ajuste auxiliar e "
                "a avaliação foi feita somente nos registros desse fluxo no teste. "
                "Os hiperparâmetros foram congelados da Tarefa 4. Como o fluxo "
                "apareceu na validação original da Tarefa 4, essa validação pode "
                "ter influenciado a escolha dos hiperparâmetros; portanto, esta é "
                "uma checagem adicional de fluxo fora do treinamento, não uma "
                "avaliação totalmente independente de todo o processo de seleção."
            )
        })

        fluxo_resultado.to_csv(
            ARTEFATOS / "fluxo_nao_visto_resultados.csv",
            index=False
        )

        print("\nInterpretação do fluxo reservado:")
        print(
            "- OK real previstos como OK:",
            int(cm_fluxo[CLASSES.index("OK"), CLASSES.index("OK")])
        )
        print(
            "- RISCO real previstos como RISCO/FALHA:",
            int(cm_fluxo[CLASSES.index("RISCO"), CLASSES.index("RISCO")])
            + int(cm_fluxo[CLASSES.index("RISCO"), CLASSES.index("FALHA")])
        )
        print(
            "- FALHA real previstos como FALHA:",
            int(cm_fluxo[CLASSES.index("FALHA"), CLASSES.index("FALHA")])
        )

with open(
    ARTEFATOS / "relatorio_fluxo_nao_visto.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        converter_json(relatorio_fluxo),
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nRelatório do fluxo salvo em:", ARTEFATOS / "relatorio_fluxo_nao_visto.json")


Fluxo reservado para a avaliação adicional: 6410|45.11.105.136
RTTs válidos no baseline A: 2150
Mediana do baseline A: 1.342342
MAD do baseline A: 0.0725074999999998
Registros desse fluxo no treino original: 1069
Registros desse fluxo na validação original: 427
Registros desse fluxo no teste: 643
Registros restantes no treino auxiliar: 15959

Avaliação adicional — fluxo excluído do treinamento
Matriz de confusão (linhas=real; colunas=previsto):


,Previsto OK,Previsto RISCO,Previsto FALHA
Real OK,486,0,0
Real RISCO,0,68,0
Real FALHA,0,0,89


              precision    recall  f1-score   support

          OK       1.00      1.00      1.00       486
       RISCO       1.00      1.00      1.00        68
       FALHA       1.00      1.00      1.00        89

    accuracy                           1.00       643
   macro avg       1.00      1.00      1.00       643
weighted avg       1.00      1.00      1.00       643

F1 macro: 1.0

Distribuição real das classes no fluxo reservado:


,count
classe,
OK,486
RISCO,68
FALHA,89


Matriz de confusão do fluxo reservado:


,Previsto OK,Previsto RISCO,Previsto FALHA
Real OK,486,0,0
Real RISCO,0,68,0
Real FALHA,0,0,89


,fluxo_id,timestamp_utc,classe,classe_prevista,acertou
1496,6410|45.11.105.136,2026-09-28 04:36:56+00:00,OK,OK,True
1497,6410|45.11.105.136,2026-09-28 04:40:48+00:00,OK,OK,True
1498,6410|45.11.105.136,2026-09-28 04:44:48+00:00,RISCO,RISCO,True
1499,6410|45.11.105.136,2026-09-28 04:48:57+00:00,RISCO,RISCO,True
1500,6410|45.11.105.136,2026-09-28 04:53:01+00:00,OK,OK,True
1501,6410|45.11.105.136,2026-09-28 04:56:52+00:00,OK,OK,True
1502,6410|45.11.105.136,2026-09-28 05:00:57+00:00,OK,OK,True
1503,6410|45.11.105.136,2026-09-28 05:04:47+00:00,FALHA,FALHA,True
1504,6410|45.11.105.136,2026-09-28 05:08:50+00:00,OK,OK,True
1505,6410|45.11.105.136,2026-09-28 05:12:47+00:00,OK,OK,True



Interpretação do fluxo reservado:
- OK real previstos como OK: 486
- RISCO real previstos como RISCO/FALHA: 68
- FALHA real previstos como FALHA: 89

Relatório do fluxo salvo em: /content/drive/MyDrive/ripe_atlas/processed/artefatos_notebooks/relatorio_fluxo_nao_visto.json


## 4. O que declarar

- [x] Se a árvore só repete a tabela de rótulo, dizer isso: o rótulo foi feito com as mesmas métricas, então a árvore está aprendendo a política, não um ticket de roteador
- [x] Se o alvo for o estado 12 minutos à frente, comparar com a regra “repetir a classe atual”. Sem ganho nessa comparação, o entregável é um detector do estado atual, não um preditor
- [x] Limitações: baseline fixo não acompanha troca de rota; rajada curta estima mal o jitter; Anchors não são a rede do campus

**Declaração sobre o alvo:** neste dataset, `classe` é a classe da própria medição/estado atual; não há coluna-alvo explícita de classe em t+12 minutos. Portanto, sem construir e validar um alvo futuro separado, o resultado deve ser apresentado como **detector/classificador do estado atual**, não como previsão de 12 minutos.

### Interpretação que deve acompanhar os números

A árvore utiliza as mesmas métricas relativas que participam da política de rotulagem da Tarefa 2. Portanto, seu desempenho mede o quanto ela consegue aproximar essa política de rotulagem; não demonstra, por si só, que identificou uma causa física de falha no roteador. O recall de FALHA e as confusões RISCO ↔ FALHA devem ser interpretados a partir da matriz exibida na seção 2.

Como `classe` representa o estado da própria medição, este experimento é um **detector/classificador do estado atual**. Não é uma previsão de 12 minutos à frente. Para afirmar capacidade preditiva seria necessário criar um alvo futuro temporal e comparar, no mesmo corte, com a regra de persistência (“repetir a classe atual”).

Limitações: (1) o baseline fixo do Período A não acompanha automaticamente uma troca de rota; (2) rajadas curtas podem estimar jitter de forma instável; (3) os fluxos Anchors representam os caminhos medidos e não toda a rede do campus; (4) a checagem de fluxo novo deve ser lida conforme o relatório de isolamento, pois um fluxo presente na validação pode ter influenciado a escolha de hiperparâmetros na Tarefa 4.


## 5. Ficha da árvore

A ficha abaixo será preenchida automaticamente na seção seguinte com os parâmetros congelados da Tarefa 4 e as métricas da única avaliação temporal. O arquivo JSON também será salvo no Drive.

| Campo | Conteúdo |
|---|---|
| Problema | Classe OK, RISCO ou FALHA pelo desvio ao baseline do fluxo |
| Unidade | Uma medição de um `fluxo_id` no Período B |
| Baseline | Período A, mediana e MAD, piso de 1.500 RTT válidos |
| Colunas | Oito features relativas/temporais listadas na ficha e no dicionário v0.4 |
| Árvore | Critério, profundidade, folhas, mínimo por folha e semente da Tarefa 4 |
| Teste (uma vez) | F1 macro, recall de FALHA e matriz 3×3 |
| O que ela não faz | Não classifica distância, não usa IP/país/rota/RTT absoluto e não opera em fluxo sem baseline |
| Como reproduzir | `requirements.txt`, notebook e pasta `processed/` |


In [15]:
# Ficha final preenchida com os resultados efetivamente calculados neste notebook.
arv_final = modelo_final.named_steps["arvore"]
parametros_finais = config_t4["parametros"]
regras_ficha = regras_final[:3]
ficha = {
    "versao": "ficha_arvore_final_v1",
    "problema": "Classificar OK, RISCO ou FALHA pelo desvio ao baseline do fluxo",
    "unidade": "Uma medição de um fluxo_id no Período B",
    "baseline": {"periodo": "A", "estatisticas": ["mediana", "MAD"],
                 "minimo_rtt_validos": 1500},
    "colunas": FEATURES,
    "colunas_proibidas_confirmadas_ausentes": ["país", "IP", "rota_id", "rtt_ms", "fluxo_id"],
    "arvore": {**resumo_arvore(modelo_final), "parametros_tarefa4": parametros_finais},
    "regras_portugues": regras_ficha,
    "teste_uma_vez": {"n_amostras": int(len(resultado_teste)),
        "acuracia": float(metricas_teste["Acurácia"]),
        "f1_macro": float(metricas_teste["F1 macro"]),
        "recall_falha": float(metricas_teste["Recall FALHA"]),
        "matriz_confusao_linhas_real_colunas_previsto": cm_teste.tolist()},
    "fluxo_nao_visto": relatorio_fluxo,
    "tipo_de_entregavel": "Detector/classificador do estado atual; classe não é alvo futuro t+12 min.",
    "o_que_nao_faz": "Não classifica distância, não identifica causa física do roteador e não opera em fluxo sem ficha de baseline.",
    "limitacoes": ["baseline fixo não acompanha troca de rota",
        "rajada curta pode estimar mal o jitter",
        "Anchors não representam toda a rede do campus",
        "fluxo presente na validação pode ter influenciado a seleção de hiperparâmetros"],
    "como_reproduzir": "requirements.txt, Tarefa5_Arvore_Final.ipynb e pasta processed/"
}
with open(ARTEFATOS / "ficha_arvore_final.json", "w", encoding="utf-8") as f:
    json.dump(converter_json(ficha), f, ensure_ascii=False, indent=2)

print("FICHA FINAL DA ÁRVORE")
display(pd.DataFrame([
    ("Problema", ficha["problema"]),
    ("Unidade", ficha["unidade"]),
    ("Baseline", "Período A; mediana/MAD; mínimo de 1.500 RTTs válidos"),
    ("Colunas", ", ".join(FEATURES)),
    ("Árvore", str(ficha["arvore"])),
    ("Teste único", f"Acurácia={metricas_teste['Acurácia']:.4%}; F1 macro={metricas_teste['F1 macro']:.4f}; recall FALHA={metricas_teste['Recall FALHA']:.4f}"),
    ("O que não faz", ficha["o_que_nao_faz"]),
    ("Como reproduzir", ficha["como_reproduzir"])
], columns=["Campo", "Conteúdo"]))

# Dicionário v0.4: somente as colunas realmente utilizadas como entrada pela árvore.
# Mantém definições e restrições originais do dicionário v0.2, sem inventar metadados.
DICIONARIO_PATH = PROCESSED / "dicionario_v0.2.csv"
if not DICIONARIO_PATH.exists():
    raise FileNotFoundError("dicionario_v0.2.csv não encontrado. Execute a Tarefa 2 primeiro.")
dicionario = pd.read_csv(DICIONARIO_PATH)
coluna_nome = "coluna" if "coluna" in dicionario.columns else dicionario.columns[0]
dicionario_v04 = dicionario[dicionario[coluna_nome].astype(str).isin(FEATURES)].copy()
if set(dicionario_v04[coluna_nome].astype(str)) != set(FEATURES):
    faltantes_dic = sorted(set(FEATURES) - set(dicionario_v04[coluna_nome].astype(str)))
    raise ValueError(f"O dicionário v0.2 não descreve todas as features: {faltantes_dic}")
# Ordena exatamente na mesma ordem usada no modelo.
dicionario_v04["_ordem"] = dicionario_v04[coluna_nome].astype(str).map({c:i for i,c in enumerate(FEATURES)})
dicionario_v04 = dicionario_v04.sort_values("_ordem").drop(columns="_ordem")
dicionario_v04.to_csv(PROCESSED / "dicionario_v0.4.csv", index=False, encoding="utf-8-sig")
print("Dicionário v0.4 salvo em:", PROCESSED / "dicionario_v0.4.csv")
display(dicionario_v04)


FICHA FINAL DA ÁRVORE


,Campo,Conteúdo
0,Problema,"Classificar OK, RISCO ou FALHA pelo desvio ao ..."
1,Unidade,Uma medição de um fluxo_id no Período B
2,Baseline,Período A; mediana/MAD; mínimo de 1.500 RTTs v...
3,Colunas,"z_robusto, aumento_pct, jitter_relativo, perda..."
4,Árvore,"{'criterion': 'gini', 'max_depth_configurado':..."
5,Teste único,Acurácia=99.9609%; F1 macro=0.9991; recall FAL...
6,O que não faz,"Não classifica distância, não identifica causa..."
7,Como reproduzir,"requirements.txt, Tarefa5_Arvore_Final.ipynb e..."


Dicionário v0.4 salvo em: /content/drive/MyDrive/ripe_atlas/processed/dicionario_v0.4.csv


,coluna,definicao,uso_restricoes
11,z_robusto,"(RTT - mediana)/(1.4826*MAD), com fallback def...",métrica relativa
12,aumento_pct,(RTT - mediana)/mediana*100,métrica relativa
13,jitter_relativo,jitter atual / jitter típico,métrica relativa
10,perda_pct,perda percentual calculada de enviados/recebidos,métrica de rotulagem
14,timeout_atual,1 se RTT ausente ou perda=100%; senão 0,métrica de evento
15,n5_timeout,timeouts nas últimas 5 medições do fluxo,métrica temporal
16,n5_aumento80,contagem dos aumentos >80% nas últimas 5,métrica temporal
17,n5_risco,contagem dos critérios de risco nas últimas 5,métrica temporal


## 6. Scrum e diário

- [ ] Board atualizado

| Integrante | O que fiz nesta tarefa | Dificuldades | O que pretendo manter/ajustar |
|---|---|---|---|
| Renata Mayumi Iuvata | Ajudei a analisar os resultados finais da árvore, conferi as métricas do teste temporal e participei da análise dos erros de classificação e do teste com fluxo não visto. | Interpretar os resultados do teste e entender as limitações da avaliação com fluxo não visto. | Continuar conferindo os resultados e melhorar a explicação das conclusões do projeto. |
| Breno Ribeiro | Fiquei responsável pela parte principal do código da árvore final, organizei a avaliação no conjunto de teste e realizei o experimento complementar com fluxo não visto, além de salvar o modelo final. | Garantir que o teste fosse realizado corretamente e que os resultados não fossem confundidos com os da validação. | Manter a organização do código e documentar melhor cada etapa da avaliação. |
| Isaque Ferreira de Jesus | — | — | — |
| Lukas Paulo Alves de Carvalho | — | — | — |
| Ruan Abner Rodrigues Morales | — | — | — |

---

---

## Rubrica — Tarefa 5 (0 a 4,0)

| Critério | Peso | Nota máxima | Nota | Observações |
|---|---:|---|---|---|
| Árvore final legível | 1,0 | Regras em português, colunas relativas, parâmetros da Tarefa 4 | | |
| Teste único | 1,5 | Matriz 3×3 em contagem, F1 por classe, F1 macro, casos curto/longo; teste não escolheu a árvore | | |
| Fluxo novo com baseline próprio | 1,0 | Ficha só no período inicial; ou limitação declarada se faltou fluxo | | |
| Ficha + diário | 0,5 | Ficha preenchida e diário de todos | | |
| **Total** | **4,0** | | **___ / 4,0** | |

In [19]:
# Salva o modelo final exatamente como veio da Tarefa 4.
# Não retreina com validação e não modifica hiperparâmetros após observar o teste.
joblib.dump(modelo_final, ARTEFATOS / "arvore_final.joblib")
resultado_teste.to_csv(ARTEFATOS / "teste_temporal_resultados.csv", index=False)
with open(ARTEFATOS / "metricas_teste_unico.json", "w", encoding="utf-8") as f:
    json.dump(converter_json({k: float(v) for k,v in metricas_teste.items()}),
              f, ensure_ascii=False, indent=2)
print("Entrega final salva em:", ARTEFATOS)
print("Arquivos adicionais: ficha_arvore_final.json, dicionario_v0.4.csv, casos_auditoria_teste.csv, relatorio_fluxo_nao_visto.json")

# ENTREGA FINAL: gera os três arquivos oficiais, sem retreinar a árvore.
# Execute a Tarefa 2 atualizada antes, para gerar o baseline com período/validade.
import sys, platform, sklearn
ENTREGA = PASTA_PROJETO / "entrega_final"
ENTREGA.mkdir(parents=True, exist_ok=True)
features_entrega = list(FEATURES)
params_entrega = modelo_final.named_steps["arvore"].get_params()
pacote_modelo = {"modelo": modelo_final, "features": features_entrega,
                 "familia": "arvores", "params": params_entrega}
joblib.dump(pacote_modelo, ENTREGA / "modelo.joblib")

baseline_entrega = pd.read_csv(BASELINE_PATH)
campos_baseline = ["fluxo_id", "mediana", "MAD", "IQR", "jitter_tipico", "perda_tipica",
                   "prop_resposta", "inicio_periodo_A_utc", "fim_periodo_A_exclusivo_utc",
                   "baseline_valido", "recomendacao_uso"]
faltantes_baseline = [c for c in campos_baseline if c not in baseline_entrega.columns]
if faltantes_baseline:
    raise ValueError(f"Reexecute a Tarefa 2 atualizada: faltam {faltantes_baseline}")
baseline_entrega.to_csv(ENTREGA / "baseline.csv", index=False, encoding="utf-8-sig")

# Exemplo real de entrada e saída no conjunto de teste, sem incluir colunas futuras.
exemplo_linha = teste.iloc[0]
exemplo_entrada = {f: (None if pd.isna(exemplo_linha[f]) else float(exemplo_linha[f])) for f in features_entrega}
exemplo_saida = str(modelo_final.predict(pd.DataFrame([exemplo_entrada], columns=features_entrega))[0])

# Documento técnico completo é mantido em um JSON-base junto ao notebook;
# as versões do ambiente e o exemplo real são atualizados a cada execução.
ARQ_BASE_META = PASTA_PROJETO / "metadados_base_entrega.json"
if not ARQ_BASE_META.exists():
    ARQ_BASE_META = Path("metadados_base_entrega.json")
if not ARQ_BASE_META.exists():
    ARQ_BASE_META = Path.cwd() / "etapas finais entrega do projeto ED2" / "metadados_base_entrega.json"
if not ARQ_BASE_META.exists():
    raise FileNotFoundError("Coloque metadados_base_entrega.json na pasta ripe_atlas do Drive ou no diretório atual.")
with open(ARQ_BASE_META, encoding="utf-8") as f:
    metadados_entrega = json.load(f)
metadados_entrega["entradas"]["ordem_features"] = features_entrega
metadados_entrega["saidas"]["classes_ordem_modelo"] = modelo_final.named_steps["arvore"].classes_.tolist()
metadados_entrega["ambiente"]["python_treinamento"] = platform.python_version()
metadados_entrega["ambiente"]["scikit_learn_treinamento"] = sklearn.__version__
metadados_entrega["ambiente"]["numpy_treinamento"] = np.__version__
metadados_entrega["ambiente"]["pandas_treinamento"] = pd.__version__
metadados_entrega["ambiente"]["joblib_treinamento"] = joblib.__version__
metadados_entrega["exemplo_real"] = {"origem":"dataset_B_com_recorte_temporal.csv, teste",
    "fluxo_id":str(exemplo_linha["fluxo_id"]), "timestamp_utc":str(exemplo_linha["timestamp_utc"]),
    "entrada":exemplo_entrada, "saida_modelo":exemplo_saida, "classe_rotulada":str(exemplo_linha["classe"])}
with open(ENTREGA / "metadados.json", "w", encoding="utf-8") as f:
    json.dump(metadados_entrega, f, ensure_ascii=False, indent=2)

# Confere contrato, ordem e previsão antes de compactar.
checagem = joblib.load(ENTREGA / "modelo.joblib")
assert set(checagem) == {"modelo", "features", "familia", "params"}
assert checagem["features"] == metadados_entrega["entradas"]["ordem_features"]
assert str(checagem["modelo"].predict(pd.DataFrame([exemplo_entrada], columns=features_entrega))[0]) == exemplo_saida
from zipfile import ZipFile, ZIP_DEFLATED
with ZipFile(ENTREGA / "NomeDoGrupo.zip", "w", compression=ZIP_DEFLATED) as z:
    for nome in ("modelo.joblib", "metadados.json", "baseline.csv"):
        z.write(ENTREGA / nome, arcname=nome)
print("Entrega validada em:", ENTREGA)



Entrega final salva em: /content/drive/MyDrive/ripe_atlas/processed/artefatos_notebooks
Arquivos adicionais: ficha_arvore_final.json, dicionario_v0.4.csv, casos_auditoria_teste.csv, relatorio_fluxo_nao_visto.json
Entrega validada em: /content/drive/MyDrive/ripe_atlas/entrega_final


In [18]:

import joblib
import pandas as pd
import sklearn
from pathlib import Path

# Caminho do modelo final
caminho = Path(
    "/content/drive/MyDrive/ripe_atlas/entrega_final/modelo.joblib"
)

# Carregar o arquivo
artefato = joblib.load(caminho)

modelo = artefato["modelo"]
features = artefato["features"]

# Exemplo real de entrada
entrada = {
    "z_robusto": 1.247650927731892,
    "aumento_pct": 9.991641474378367,
    "jitter_relativo": 2.8221049395699787,
    "perda_pct": 0.0,
    "timeout_atual": 0,
    "n5_timeout": 0,
    "n5_aumento80": 0,
    "n5_risco": 1
}

# Garantir a ordem correta das features
dados = pd.DataFrame([entrada])[features]

# Executar a previsão
previsao = modelo.predict(dados)

print("Versão scikit-learn:", sklearn.__version__)
print("Features utilizadas:", features)
print("Previsão do modelo:", previsao[0])

assert previsao[0] == "OK", "Previsão diferente do exemplo documentado!"
print("TESTE CONCLUÍDO COM SUCESSO!")


Versão scikit-learn: 1.6.1
Features utilizadas: ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']
Previsão do modelo: OK
TESTE CONCLUÍDO COM SUCESSO!
